# Data acquisition via Web scraping

This notebook demonstrates the use of Selenium to web scrape data from a National Highways website. It outlines the methodological and logical approach taken to extract information from an external source. The exercise was undertaken to evaluate and compare this data acquisition method with the use of an API. Following careful consideration, an API-based approach was selected for the final data pipeline.

In [4]:
# import requests to handle HTTP requests for fetching web pages and APIs
import requests
# imports load_dotenv function from the dotenv library to load environment variables from a .env file
from dotenv import load_dotenv
#imports os library for interacting with the operating system
import os
# import re to handle regular expressions
import re
# import logging to log messages for debugging and tracking the execution of the script
import logging
# import time for sleep intervals between actions to allow page elements to load properly
import time
# import random to introduce randomness in sleep intervals to mimic human behavior and avoid being blocked by the website
import random
# import typing for type hints to improve code readability and maintainability
from typing import Callable, Tuple, Type
# import pandas for data manipulation and analysis, especially for working with DataFrames
import pandas as pd
# import selenium for web scraping to automate browser interactions and extract data from web pages
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

## Web scraping with Selenium

In this section, web scraping was performed on the National Highways website. Selenium was used as the website’s dynamic structure meant that data could not be extracted using BeautifulSoup. Selenium allowed the rendered content to be accessed and the required data to be successfully collected.

In [ ]:
# Configure global logging: INFO level with timestamp, severity, module, and message.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

# Create a module-specific logger for traceable, structured logging.
logger = logging.getLogger(__name__)

In [ ]:
# Defines a browser-like User-Agent header to reduce the likelihood of request blocking by web server bot protection mechanisms
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

# Specifies the standard endpoint for retrieving the website's robots.txt file, which outlines permitted scraping rules
robot_url = "https://www.nationalhighways.co.uk/robots.txt"

# Sends a GET request to the robots.txt URL with custom headers; SSL verification is disabled due to local certificate issues
r = requests.get(robot_url, headers=headers, verify=False, timeout=30)


# Logs the HTTP status code
logger.info("robots.txt request returned status code %s", r.status_code)

In [5]:
# Retrieves the API key from environment variables
FOLDER_PATH = os.getenv("FOLDER_PATH")  # Path to the Edge WebDriver executable, specified in the .env file

In [ ]:
EDGE_DRIVER_PATH = FOLDER_PATH  # Path to the Edge WebDriver executable, specified in the .env file

In [ ]:
# Proxy support is intentionally DISABLED for this task.
# The code supports proxies if required, but the scraper
# runs via a direct connection due to low request volume and the
# public-sector nature of the target website.

PROXIES = [
    None  # Direct connection (default and intentional)
    # "proxy1",
    # "proxy2",
]

In [ ]:
# =========================
# Rate limiting & retry utilities
# =========================

def polite_sleep(base: float = 1.0, jitter: float = 0.5) -> None:

    """
    Pause the program for a short amount of time.

    This is used to slow down requests so the website is not
    overwhelmed and to make the scraper behave more like a human.
    A small random delay is added to avoid predictable timing.
    """

    delay = base + random.uniform(0, jitter)
    time.sleep(delay)


def retry(
    fn: Callable,
    retries: int = 3,
    delay: float = 1.0,
    backoff: float = 2.0,
    exceptions: Tuple[Type[Exception], ...] = (Exception,)
):

    """
    Try to run a function again if it fails.

    If an error occurs, the function will wait for a short time
    and then try again. Each retry waits longer than the last.
    This helps handle temporary issues like slow page loading.
    """

    attempt = 0
    current_delay = delay

    while attempt < retries:
        try:
            return fn()
        except exceptions as e:
            attempt += 1
            if attempt >= retries:
                raise
            logger.warning(
                "Retry %d/%d after error: %s",
                attempt, retries, e
            )
            time.sleep(current_delay)
            current_delay *= backoff

# =========================
# Build WebDriver with optional proxy support
# =========================

def build_driver(
    headless: bool = False,
    proxy: str | None = None
) -> webdriver.Edge:

    """
    Create and configure a new Edge browser for Selenium.

    This function sets up the browser window, optional headless mode,
    and optional proxy support. The browser created here is used to
    load and interact with web pages.
    """

    opts = Options()
    opts.add_argument("--window-size=1400,900")

    if headless:
        opts.add_argument("--headless=new")

    # Proxy capability (not enabled in current execution)
    if proxy:
        opts.add_argument(f"--proxy-server={proxy}")
        logger.info("Proxy configured: %s", proxy)

    service = Service(EDGE_DRIVER_PATH)
    return webdriver.Edge(service=service, options=opts)


# =========================
# Page preparation
# =========================

def accept_cookies_if_present(driver: webdriver.Edge) -> None:

    """
    Accepts the cookie consent banner if it appears.
    
    If no cookie banner is found, the function simply does nothing.
    """

    selectors = [
        (By.XPATH, "//button[contains(., 'Accept') or contains(., 'ACCEPT')]"),
        (By.XPATH, "//a[contains(., 'Accept') or contains(., 'ACCEPT')]"),
    ]

    for by, sel in selectors:
        try:
            btn = driver.find_element(by, sel)
            driver.execute_script("arguments[0].click();", btn)
            polite_sleep(0.5)
            return
        except Exception:
            pass

def open_page_and_accept_cookies(
    driver: webdriver.Edge,
    url: str,
    timeout: int = 30
) -> WebDriverWait:
    
    """
    Open the target web page and handle cookies.

    This function loads the given URL, waits for the page to be ready,
    and attempts to accept cookies if prompted. It returns a wait
    object used for later page interactions.
    """

    driver.get(url)
    wait = WebDriverWait(driver, timeout)
    accept_cookies_if_present(driver)
    return wait


def configure_results_view(driver: webdriver.Edge, wait: WebDriverWait) -> None:

    """
    Change the page settings to show more results at once.

    This function increases the number
    of results displayed to reduce the need for scrolling
    or pagination.
    """

    try:
        wait.until(EC.presence_of_element_located((By.ID, "search-results")))

        try:
            per_page = driver.find_element(By.ID, "perpage")
            Select(per_page).select_by_value("9999")
            polite_sleep(2.0)
        except NoSuchElementException:
            pass

    except TimeoutException:
        logger.warning("search-results container did not load")


# =========================
# Lazy loading support
# =========================

def load_all_results(
    driver: webdriver.Edge,
    wait: WebDriverWait,
    scroll_pause: float = 0.5
) -> None:
    
    """
    Scroll the page to force all results to load.


    This function scrolls both the page to ensure all closure records are visible.
    """

    try:
        search_el = wait.until(
            EC.presence_of_element_located((By.ID, "search-results"))
        )
    except TimeoutException:
        return

    for _ in range(8):
        driver.execute_script("window.scrollBy(0, 900)")
        polite_sleep(scroll_pause)

    for _ in range(12):
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight;",
            search_el
        )
        polite_sleep(scroll_pause)


# =========================
# Extraction
# =========================

def extract_raw_records(driver: webdriver.Edge) -> list[str]:
    
    """
    Collect raw text blocks that are closure records.

    This function scans the page for elements containing dates,
    removes duplicates, and returns the unprocessed text
    for each road closure.
    """

    records: list[str] = []
    seen: set[str] = set()

    try:
        candidates = driver.find_elements(
            By.XPATH,
            "//*[@id='search-results']//*[contains(., '/') and contains(., '202')]"
        )
    except Exception:
        return records

    for el in candidates:
        try:
            container = el.find_element(
                By.XPATH, "./ancestor::*[self::div or self::li][1]"
            )
            txt = container.text.strip()

            if txt and len(re.findall(r"\b\d{2}/\d{2}/\d{4}\b", txt)) >= 2:
                if txt not in seen:
                    seen.add(txt)
                    records.append(txt)

        except Exception:
            continue

    return records

# =========================
# Parsing 
# =========================

def parse_record(raw: str) -> pd.Series:
    
    """
    Convert a raw text record into structured data.

    This function extracts useful information such as road name,
    dates, times, and title from a block of text and returns it
    in a structured format suitable for analysis.
    """

    raw = "" if raw is None else raw

    dates = re.findall(r"\b\d{2}/\d{2}/\d{4}\b", raw)
    times = re.findall(r"\b\d{1,2}:\d{2}\b", raw)

    m = re.search(r"\b(M\d+|A\d+[A-Z0-9()]*)\b", raw, flags=re.I)
    road = m.group(1).upper() if m else None

    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    title = next((ln for ln in lines if "closure" in ln.lower()), None)

    return pd.Series({
        "road": road,
        "title": title,
        "start_date": dates[0] if len(dates) > 0 else None,
        "start_time": times[0] if len(times) > 0 else None,
        "end_date": dates[1] if len(dates) > 1 else None,
        "end_time": times[1] if len(times) > 1 else None,
    })


# =========================
# Main scraping pipeline
# =========================

def scrape_national_highways_closures(
    url: str,
    timeout: int = 30,
    headless: bool = False
) -> pd.DataFrame:

    """
    Run the full web scraping process and return the results.

    This function controls the entire workflow:
    opening the website, loading all results, extracting
    closure records, parsing them, and returning the data
    as a structured table.
    """

    proxy = PROXIES[0]  # Explicitly None by design
    driver = build_driver(headless=headless, proxy=proxy)

    try:
        wait = retry(
            lambda: open_page_and_accept_cookies(driver, url, timeout),
            retries=3,
            delay=2.0
        )

        polite_sleep(1.5)
        configure_results_view(driver, wait)
        load_all_results(driver, wait)

        raw_records = retry(
            lambda: extract_raw_records(driver),
            retries=2,
            delay=1.0
        )

        parsed = [parse_record(r) for r in raw_records]
        return pd.DataFrame(parsed)

    finally:
        driver.quit()



In [ ]:
# Call a custom function to scrape National Highways closure data and put into a dataframe
df = scrape_national_highways_closures(URL, timeout=30, headless=False)
df.head(10)

## Exploratory Data Analysis

In this section, exploratory data analysis (EDA) was conducted to assess the quality and structure of the scraped data. The EDA revealed several data inconsistencies, including missing values within rows, which required further cleaning and preprocessing prior to analysis.

In [ ]:
# Displays info about the DataFrame, including column names, data types, and non-null counts
df.info()

In [ ]:
# Gives a description of the DataFrame
df.describe

In [ ]:
# Give all unique values in the "road" column to see which roads are affected by closures
df["road"].unique()

In [ ]:
# Give all unique values in the "title" column
df["title"].unique()